In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# loading datasets
fews_dt =  pd.read_csv("data/output/fews_cleaned.csv")
wfp_dt = pd.read_csv("data/output/wfp_cleaned.csv")

In [3]:
fews_dt.head() # loading the top data in fews_cleaned

,country,market,admin_1,longitude,latitude,cpcv2,source_document,price_type,product_source,unit,unit_type,currency,date,commodity,price,price_missing
0,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-07-14,bread,NaN,1
1,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-08-15,bread,NaN,1
2,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-09-15,bread,NaN,1
3,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2008-12-16,bread,NaN,1
4,Nigeria,aba,Abia,7.37063,5.10719,P23490AA,Famine Early Warning Systems Network (FEWS NET...,Retail,Local,ea,Item,NGN,2009-01-23,bread,NaN,1


In [4]:
wfp_dt.head() # loading the top data in wfp_cleaned

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,currency,price,usdprice,price_type
0,2002-01-15,Katsina,Jibia,jibia(cbm),1038,13.08,7.24,cereals and tubers,maize,51,kg,actual,NGN,175.92,1.54,Wholesale
1,2002-01-15,Katsina,Jibia,jibia(cbm),1038,13.08,7.24,cereals and tubers,millet,73,kg,actual,NGN,150.18,1.31,Wholesale
2,2002-01-15,Katsina,Jibia,jibia(cbm),1038,13.08,7.24,cereals and tubers,rice (imported),64,kg,actual,NGN,358.70,3.14,Wholesale
3,2002-01-15,Katsina,Jibia,jibia(cbm),1038,13.08,7.24,cereals and tubers,sorghum,65,kg,actual,NGN,155.61,1.36,Wholesale
4,2002-01-15,Katsina,Jibia,jibia(cbm),1038,13.08,7.24,pulses and nuts,beans (niebe),120,kg,actual,NGN,196.87,1.72,Wholesale


In [5]:
print("FEWS unique commodity:", fews_dt['commodity'].unique())
print() # checking for mapped commodity in fews and wfp
print("WFP unique commodity:", wfp_dt['commodity'].unique())

FEWS unique commodity: ['bread' 'cowpeas (brown)' 'cowpeas (white)' 'diesel' 'gari (white)'
 'gari (yellow)' 'gasoline' 'groundnuts (shelled)' 'maize grain (white)'
 'maize grain (yellow)' 'millet (pearl)' 'palm oil (refined)'
 'rice (5% broken)' 'rice (milled)' 'sorghum (brown)' 'sorghum (white)'
 'yams' 'cattle (male)' 'goats (male)' 'sheep (male)']

WFP unique commodity: ['maize' 'millet' 'rice (imported)' 'sorghum' 'beans (niebe)' 'wheat'
 'maize (white)' 'sorghum (white)' 'rice (milled, local)' 'bread'
 'cassava meal (gari, yellow)' 'gari (white)' 'maize (yellow)'
 'rice (local)' 'sorghum (brown)' 'yam (abuja)' 'fuel (diesel)'
 'fuel (petrol-gasoline)' 'oil (palm)' 'cowpeas (brown)' 'cowpeas (white)'
 'yam' 'groundnuts (shelled)' 'maize flour' 'meat (beef)' 'meat (goat)'
 'milk (powder)' 'oil (vegetable)' 'beans (red)' 'beans (white)'
 'groundnuts' 'onions' 'fish' 'eggs' 'bananas' 'oranges' 'spinach'
 'watermelons' 'cowpeas' 'tomatoes' 'salt' 'sugar'
 'seasoning (maggi cube)']


In [6]:

# Standardizes variations across systems (e.g., matching 
# FEWS' 'maize grain (white)' and WFP's 'maize (white)' to 'maize (white)').
#  Many-to-One Resolution: Safely handles reporting asymmetry, such as mapping both generic 'yam' and specific 'yam (abuja)' from WFP to a uniform base.

commodity_map = pd.DataFrame([
    {'fews_commodity': 'bread', 'wfp_commodity': 'bread', 'mapped_name': 'bread'},
    {'fews_commodity': 'cowpeas (brown)', 'wfp_commodity': 'cowpeas (brown)', 'mapped_name': 'cowpeas (brown)'},
    {'fews_commodity': 'cowpeas (white)', 'wfp_commodity': 'cowpeas (white)', 'mapped_name': 'cowpeas (white)'},
    {'fews_commodity': 'gari (white)', 'wfp_commodity': 'gari (white)', 'mapped_name': 'gari (white)'},
    {'fews_commodity': 'gari (yellow)', 'wfp_commodity': 'cassava meal (gari, yellow)', 'mapped_name': 'gari (yellow)'},
    {'fews_commodity': 'groundnuts (shelled)', 'wfp_commodity': 'groundnuts (shelled)', 'mapped_name': 'groundnuts (shelled)'},
    {'fews_commodity': 'sorghum (brown)', 'wfp_commodity': 'sorghum (brown)', 'mapped_name': 'sorghum (brown)'},
    {'fews_commodity': 'sorghum (white)', 'wfp_commodity': 'sorghum (white)', 'mapped_name': 'sorghum (white)'},
    {'fews_commodity': 'maize grain (white)', 'wfp_commodity': 'maize (white)', 'mapped_name': 'maize (white)'},
    {'fews_commodity': 'maize grain (yellow)', 'wfp_commodity': 'maize (yellow)', 'mapped_name': 'maize (yellow)'},
    {'fews_commodity': 'millet (pearl)', 'wfp_commodity': 'millet', 'mapped_name': 'millet'},
    {'fews_commodity': 'rice (milled)', 'wfp_commodity': 'rice (milled, local)', 'mapped_name': 'rice (milled)'},
    {'fews_commodity': 'yams', 'wfp_commodity': 'yam', 'mapped_name': 'yams'},
    {'fews_commodity': 'yams', 'wfp_commodity': 'yam (abuja)', 'mapped_name': 'yams'},
    {'fews_commodity': 'palm oil (refined)', 'wfp_commodity': 'oil (palm)', 'mapped_name': 'palm oil'},
    {'fews_commodity': 'diesel', 'wfp_commodity': 'fuel (diesel)', 'mapped_name': 'diesel'},
    {'fews_commodity': 'gasoline', 'wfp_commodity': 'fuel (petrol-gasoline)', 'mapped_name': 'gasoline'},
])
# mapped commodity in fews to wfp 
# Save to mappings folder
commodity_map.to_csv('mapping/commodity_map.csv', index=False)
print("Saved commodity_map.csv")
print(commodity_map)

Saved commodity_map.csv
          fews_commodity                wfp_commodity           mapped_name
0                  bread                        bread                 bread
1        cowpeas (brown)              cowpeas (brown)       cowpeas (brown)
2        cowpeas (white)              cowpeas (white)       cowpeas (white)
3           gari (white)                 gari (white)          gari (white)
4          gari (yellow)  cassava meal (gari, yellow)         gari (yellow)
5   groundnuts (shelled)         groundnuts (shelled)  groundnuts (shelled)
6        sorghum (brown)              sorghum (brown)       sorghum (brown)
7        sorghum (white)              sorghum (white)       sorghum (white)
8    maize grain (white)                maize (white)         maize (white)
9   maize grain (yellow)               maize (yellow)        maize (yellow)
10        millet (pearl)                       millet                millet
11         rice (milled)         rice (milled, local)         ri

In [7]:

# Standardizes grams and bulk metrics (e.g., 50 kg) 
# to a clean 'kg' base unit using a divisor multiplier.
# Type Safety (The 'convertible' flag): Segregates liquids (litres), 
# counts (tubers), and pieces (each) to guarantee that downstream DuckDB 
# reconciliation joins only compare identical semantic categories.

unit_conversion = pd.DataFrame([
    # KG variants - conversion factor to get price per kg
    {'unit': 'kg', 'conversion_factor': 1, 'base_unit': 'kg', 'convertible': True},
    {'unit': '100 kg', 'conversion_factor': 100, 'base_unit': 'kg', 'convertible': True},
    {'unit': '50 kg', 'conversion_factor': 50, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.6 kg', 'conversion_factor': 2.6, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.8 kg', 'conversion_factor': 2.8, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.7 kg', 'conversion_factor': 2.7, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.5 kg', 'conversion_factor': 2.5, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.1 kg', 'conversion_factor': 2.1, 'base_unit': 'kg', 'convertible': True},
    {'unit': '2.2 kg', 'conversion_factor': 2.2, 'base_unit': 'kg', 'convertible': True},
    {'unit': '0.5 kg', 'conversion_factor': 0.5, 'base_unit': 'kg', 'convertible': True},
    {'unit': '1.3 kg', 'conversion_factor': 1.3, 'base_unit': 'kg', 'convertible': True},
    # Gram variants - convert to kg
    {'unit': '400 g', 'conversion_factor': 0.4, 'base_unit': 'kg', 'convertible': True},
    {'unit': '300 g', 'conversion_factor': 0.3, 'base_unit': 'kg', 'convertible': True},
    {'unit': '250 g', 'conversion_factor': 0.25, 'base_unit': 'kg', 'convertible': True},
    {'unit': '500 g', 'conversion_factor': 0.5, 'base_unit': 'kg', 'convertible': True},
    # Litres - leave as is, compare litre-to-litre only
    {'unit': 'l', 'conversion_factor': 1, 'base_unit': 'l', 'convertible': False},
    {'unit': '30 l', 'conversion_factor': 30, 'base_unit': 'l', 'convertible': False},
    {'unit': '100 l', 'conversion_factor': 100, 'base_unit': 'l', 'convertible': False},
    # Tubers - leave as is, compare tuber-to-tuber only
    {'unit': '100 tubers', 'conversion_factor': 100, 'base_unit': 'tubers', 'convertible': False},
    {'unit': '60 tubers', 'conversion_factor': 60, 'base_unit': 'tubers', 'convertible': False},
    # Per piece - leave as is
    {'unit': 'ea', 'conversion_factor': 1, 'base_unit': 'piece', 'convertible': False},
    {'unit': 'unit', 'conversion_factor': 1, 'base_unit': 'piece', 'convertible': False},
    {'unit': '30 pcs', 'conversion_factor': 30, 'base_unit': 'piece', 'convertible': False},
])

# Save to mappings folder
unit_conversion.to_csv('mapping/unit_conversion.csv', index=False)
print("Saved unit_conversion.csv")
print(unit_conversion)

Saved unit_conversion.csv
          unit  conversion_factor base_unit  convertible
0           kg               1.00        kg         True
1       100 kg             100.00        kg         True
2        50 kg              50.00        kg         True
3       2.6 kg               2.60        kg         True
4       2.8 kg               2.80        kg         True
5       2.7 kg               2.70        kg         True
6       2.5 kg               2.50        kg         True
7       2.1 kg               2.10        kg         True
8       2.2 kg               2.20        kg         True
9       0.5 kg               0.50        kg         True
10      1.3 kg               1.30        kg         True
11       400 g               0.40        kg         True
12       300 g               0.30        kg         True
13       250 g               0.25        kg         True
14       500 g               0.50        kg         True
15           l               1.00         l        False
16   

In [8]:
market_map = pd.DataFrame([
    # Overlapping markets - exist in both sources
    {'fews_market': 'aba', 'wfp_market': 'aba', 'status': 'overlap'},
    {'fews_market': 'biu', 'wfp_market': 'biu', 'status': 'overlap'},
    {'fews_market': 'damaturu', 'wfp_market': 'damaturu', 'status': 'overlap'},
    {'fews_market': 'dandume', 'wfp_market': 'dandume', 'status': 'overlap'},
    {'fews_market': 'giwa', 'wfp_market': 'giwa', 'status': 'overlap'},
    {'fews_market': 'gombe', 'wfp_market': 'gombe', 'status': 'overlap'},
    {'fews_market': 'gujungu', 'wfp_market': 'gujungu', 'status': 'overlap'},
    {'fews_market': 'gwandu', 'wfp_market': 'gwandu', 'status': 'overlap'},
    {'fews_market': 'maiduguri', 'wfp_market': 'maiduguri', 'status': 'overlap'},
    {'fews_market': 'mubi', 'wfp_market': 'mubi', 'status': 'overlap'},
    {'fews_market': 'potiskum', 'wfp_market': 'potiskum', 'status': 'overlap'},
    {'fews_market': 'saminaka', 'wfp_market': 'saminaka', 'status': 'overlap'},
    # FEWS only markets - no WFP counterpart
    {'fews_market': 'ibadan', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'ilela', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'kano', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'kauranamoda', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'lagos', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'maiadua', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'maigatari', 'wfp_market': None, 'status': 'fews_only'},
    {'fews_market': 'port-harcourt', 'wfp_market': None, 'status': 'fews_only'},
])

# Save to mappings folder
market_map.to_csv('mapping/market_map.csv', index=False)
print("Saved market_map.csv")
print(market_map)

Saved market_map.csv
      fews_market wfp_market     status
0             aba        aba    overlap
1             biu        biu    overlap
2        damaturu   damaturu    overlap
3         dandume    dandume    overlap
4            giwa       giwa    overlap
5           gombe      gombe    overlap
6         gujungu    gujungu    overlap
7          gwandu     gwandu    overlap
8       maiduguri  maiduguri    overlap
9            mubi       mubi    overlap
10       potiskum   potiskum    overlap
11       saminaka   saminaka    overlap
12         ibadan       None  fews_only
13          ilela       None  fews_only
14           kano       None  fews_only
15    kauranamoda       None  fews_only
16          lagos       None  fews_only
17        maiadua       None  fews_only
18      maigatari       None  fews_only
19  port-harcourt       None  fews_only
